# A tiny model to explain the QML principles using PennyLane

_This notebook explores the creation and use of a very simple model in **PennyLane and PyTorch**_.

**By:** Jacob Cybulski ([website](https://jacobcybulski.com/))<br>
**Date:** Mar 26, 2025<br>
**Updated:** Jun 2, 2026<br>
**Aims:** To develop a tiny quantum model in PennyLane to explain QML principles<br/>
**Prerequisites:** We will assume your knowledge of *quantum computing*, *machine learning* and *Python*<br>
**License:** 
This project is licensed under the [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.txt)<br>
**Changes:** All changes to this code must be listed at the bottom of this notebook

## Libraries

In [1]:
### General libraries
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
### PennyLane libraries
import pennylane as qml
from pennylane import numpy as np

## Explore a PennyLane tiny quantum model

We will first identify what quantum devices and gradient support we have, then we will show the principles of working with PennyLane, then prepare data for model training and testing, create a quantum model, and finally train it and test it.

### Define device to compute on
*We have a variety of different types of quantum devices.*
- **default.qubit:** used for state vector simulations
- **lightning.qubit:** high performance qubit (written in C++)
- **lightning.gpu:** a GPU state vector qubit simulation (NVIDIA)
- **default.tensor:** a state vector simulator based on tensor networks
- **qiskit.remote:** a Qiskit device accessible via *qiskit_ibm_runtime* interface
- **default.mixed:** a mixed state simulator of Qiskit based quantum circuits
- *And many more (over 40 different devices)*

In [3]:
### Select a device for PL operation

# Quantum simulator
sim = 'default.qubit'

### Show QML principles with PennyLane

### Set up the data
*Note that weights require gradients and inputs are just plain values.*

In [4]:
### Test data
#   Note that 'np' is from PennyLane, so numpy array is automatically a tensor!

# How many 'wires' should we have
n_qubits = 2

# Plain values for input
inputs = np.array([np.pi*0.7], requires_grad=False)

# Parameters with gradients for weights
weights = np.array([[1.7, 1.29], [-0.5, 0.5]], requires_grad=True)

In [5]:
# This is our data
inputs

tensor([2.19911486], requires_grad=False)

In [6]:
# These are weights
weights

tensor([[ 1.7 ,  1.29],
        [-0.5 ,  0.5 ]], requires_grad=True)

### Define a circuit

In [7]:
### Simple circuit creation and execution
dev = qml.device(sim, wires=n_qubits)

@qml.qnode(dev, shots=1000)
def qnn_model(weights, x):
    
    # Data encoding
    qml.RY(x[0], wires=0)
    qml.Barrier()
    
    # Ansatz
    qml.RX(weights[0, 0], wires=0)
    qml.RY(weights[0, 1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(weights[1, 0], wires=0)
    qml.RY(weights[1, 1], wires=1)
    qml.Barrier()

    # Measurement
    #return qml.probs(wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(m)) for m in range(n_qubits)]

In [8]:
### Plot the circuit
print('Circuit:')
print(qml.draw(qnn_model)(weights, inputs), '\n')

Circuit:
0: ──RY(2.20)──||──RX(1.70)─╭●──RY(-0.50)──||─┤  <Z>
1: ────────────||──RY(1.29)─╰X──RY(0.50)───||─┤  <Z> 



#### Apply the circuit to some data and see the results
*In PennyLane, the order of qubits (wires) follows a __big-endian convention__, where:<br>
the wire with the lowest index (e.g., wire 0) corresponds to the most significant bit (MSB) in the resulting state vector.<br>
This is opposite to Qiskit which uses a __little-endian convention__.*

In [9]:
### Run the circuit
results1 = qnn_model(weights, inputs)
results1

[array(0.444), array(-0.43)]

#### Let's run this circuit a few times
*Observe results, why do you think we get different results each time?*

In [10]:
results2 = qnn_model(weights, inputs)
results2

[array(0.438), array(-0.398)]

In [11]:
results3 = qnn_model(weights, inputs)
results3

[array(0.432), array(-0.444)]

#### Show the gradients at the end of the process
*Gradients are shown in the form of the Jacobian matrix J(i, j) for the circuit weights,
where each term is the partial derivative of the probability $P_{ij}$ of measuring state $\vert ij \rangle$ with respect to the k-th weight, $w_k$.*

In [12]:
qml.gradients.param_shift(qnn_model)(weights, inputs)

[array([[ 0.514,  0.097],
        [-0.651, -0.037]]),
 array([[ 0.113, -0.188],
        [ 0.   , -0.868]])]

#### Now let's pass multiple inputs to the circuit
*__Warning:__ if applying the circuit to multiple inputs is confusing, just do it in a loop one input at a time.*<br> 
*Note that the result is produced in the form to support PennyLane gradient calculations.*<br>
*However, this format is not compatible with what we are used to in other ML packages.*

In [13]:
### Prepare a tensor of multiple inputs
multi_inputs = np.stack([inputs, inputs, inputs, inputs, inputs])
multi_inputs

tensor([[2.19911486],
        [2.19911486],
        [2.19911486],
        [2.19911486],
        [2.19911486]], requires_grad=False)

In [14]:
### Get the results
results4 = qnn_model(weights, multi_inputs)
results4

[array(0.454), array(-0.448)]

__*What happened here?*__<br>
*We expect the circuit to be called $n$ times and return $n$ sets of 4 measurement outcomes each.<br>
However, PL uses "parameter broadcasting", where the qnode code is run once only!<br>
The first objective of this run is to create the circuit's abstract representation, called quantum tape.<br>
The second objective is to create a tensor of values to be assigned to each parameter!<br>
Each tensor must be of the same size, which is called the batch size.<br>
Then, PL will assign values to each parameter and execute the circuit the batch size times.<br>
So, consider what is inputs[0] and inputs[1] in the previous case, how many results will be produced?<br> 
How should the circuit be changed, to provide $n$ correct values to each input parameter?*

In [15]:
### Another version of the circuit for batched inputs
@qml.qnode(dev)
def qnn_model_batched(weights, x):
    
    # Data encoding
    qml.RY(x[:, 0], wires=0)
    qml.Barrier()
    
    # Ansatz
    qml.RX(weights[0, 0], wires=0)
    qml.RY(weights[0, 1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(weights[1, 0], wires=0)
    qml.RY(weights[1, 1], wires=1)
    qml.Barrier()
    
    # Measurement
    #return qml.probs(wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(m)) for m in range(n_qubits)]

In [16]:
### Plot the circuit
print('Batched circuit:')
print(qml.draw(qnn_model_batched)(weights, multi_inputs), '\n');

Batched circuit:
0: ──RY(M0)──||──RX(1.70)─╭●──RY(-0.50)──||─┤  <Z>
1: ──────────||──RY(1.29)─╰X──RY(0.50)───||─┤  <Z>

M0 = 
[2.19911486 2.19911486 2.19911486 2.19911486 2.19911486] 



In [17]:
### Get the results
results5 = qnn_model_batched(weights, multi_inputs)
results5

[tensor([0.43913463, 0.43913463, 0.43913463, 0.43913463, 0.43913463], requires_grad=True),
 tensor([-0.4422309, -0.4422309, -0.4422309, -0.4422309, -0.4422309], requires_grad=True)]

In [18]:
### Transpose the results
results6 = np.transpose(np.stack(results5)).numpy()
results6

array([[ 0.43913463, -0.4422309 ],
       [ 0.43913463, -0.4422309 ],
       [ 0.43913463, -0.4422309 ],
       [ 0.43913463, -0.4422309 ],
       [ 0.43913463, -0.4422309 ]])

__*This is now what we expected to get!*__

## What's next?
At this point you can try the _**s01_simple_model**_ exercise.

## Modifications (do not remove)
Under the [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.txt) license, if you perform any changes to this notebook, please list them here, adding a note with your name, contact details, date and changes to the code.

- [Jacob Cybulski](http://jacobcybulski.com) (2024, 1 Oct): The author of this notebook added this section to record all code changes
- [Jacob Cybulski](http://jacobcybulski.com) (2026, 9 Apr): Now also compatible with...<br>
  pennylane                 0.44.1<br>
  pennylane_lightning       0.44.0<br>
  torch                     2.11.0+cpu<br>
  torchaudio                2.11.0<br>
  torchvision               0.26.0+cpu<br>
- [Jacob Cybulski](http://jacobcybulski.com) (2026, 2 Jun): Adaptation for Deakin tutorial

## Systems in use (Linux)

In [19]:
!pip list | grep -e pennylane -e torch

pennylane                 0.43.2
pennylane_lightning       0.43.0
torch                     2.6.0+cu126
torch-geometric           2.6.1
torchaudio                2.6.0+cu126
torcheval                 0.0.7
torchsummary              1.5.1
torchvision               0.21.0+cu126
